# 04 - HydroServer Quality Control Demo

## Setup and Creation Controls
This short notebook runs local QC first, then shows the optional HydroServerQualityControl pattern for an authenticated datastream.

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path

import pandas as pd

try:
    from hydroserverpy import HydroServer
except Exception as exc:
    HydroServer = None
    print(f"hydroserverpy is not available yet: {exc}")

HYDROSERVER_HOST = "https://playground.hydroserver.org"
WORKSPACE_NAME = "hydroserver_uganda_demo"
WORKSPACE_IS_PRIVATE = False

# Choose one: "anonymous" or "api_key".
AUTH_METHOD = "api_key"
HYDROSERVER_API_KEY = ""  # Keep secrets out of saved notebooks. Paste only when prompted.

# Facilitator controls. Defaults keep notebooks safe for anonymous/local runs.
CREATE_WORKSPACE_IF_MISSING = False
DELETE_CREATED_RESOURCES_AT_END = False
DEMO_RESOURCE_PREFIX = "Uganda Demo"
DEMO_RUN_SUFFIX = ""

# Google Colab/local path support. Leave blank unless data is somewhere custom.
DATA_DIR_OVERRIDE = ""


def resolve_data_dir(data_dir_override=""):
    candidates = []
    if data_dir_override:
        candidates.append(Path(data_dir_override).expanduser())
    candidates.extend([
        Path("data"),
        Path("../data"),
        Path("hydroserver_workshop/data"),
        Path("../hydroserver_workshop/data"),
        Path("/content/hydroserver_workshop/data"),
        Path("/content/data"),
    ])
    for candidate in candidates:
        if candidate.exists() and (candidate / "Uganda_Hydroweb.csv").exists():
            return candidate
    searched = "\n".join(f"- {candidate}" for candidate in candidates)
    raise FileNotFoundError(
        "Could not find the workshop data folder. Upload hydroserver_workshop/data, "
        "upload data/ beside the notebook, or set DATA_DIR_OVERRIDE. Searched:\n"
        f"{searched}"
    )


DATA_DIR = resolve_data_dir(DATA_DIR_OVERRIDE)
STATION_CATALOG_CSV = DATA_DIR / "Uganda_Hydroweb.csv"
SELECTED_STATION_CSV = DATA_DIR / "uganda_selected_station.csv"
HYDROWEB_DIR = DATA_DIR / "hydroweb"
GEOGLOWS_DIR = DATA_DIR / "geoglows"
STREAMFLOW_CSV = DATA_DIR / "sample_streamflow_observations.csv"
FORECAST_CSV = DATA_DIR / "sample_forecast_timeseries.csv"

demo_run_suffix = DEMO_RUN_SUFFIX or datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
demo_resource_prefix = f"{DEMO_RESOURCE_PREFIX} {demo_run_suffix}"

print(f"HydroServer host: {HYDROSERVER_HOST}")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Authentication mode: {AUTH_METHOD}")
print(f"Data directory: {DATA_DIR}")

QUALITY_DATASTREAM_ID = ""  # Optional existing datastream UUID for HydroServer-backed QC.
UPLOAD_QUALITY_CONTROLLED_RESULTS = False

HydroServer host: https://playground.hydroserver.org
Workspace: hydroserver_uganda_demo
Authentication mode: api_key
Data directory: ../data


## Connect to HydroServer

In [3]:
def _prompt_if_needed(value, prompt):
    return value if value else getpass(prompt)

hs_api = None

if HydroServer is None:
    print("Install hydroserverpy before connecting to HydroServer.")
elif AUTH_METHOD == "anonymous":
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST)
        print("Connected anonymously. Anonymous mode can read public data but cannot create or upload resources.")
    except Exception as exc:
        print(f"Anonymous connection failed: {exc}")
elif AUTH_METHOD == "api_key":
    api_key = _prompt_if_needed(HYDROSERVER_API_KEY, "HydroServer API key: ")
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST, apikey=api_key)
        print("Connected with API-key authentication.")
    except Exception as exc:
        print(f"API-key connection failed: {exc}")
else:
    raise ValueError("AUTH_METHOD must be 'anonymous' or 'api_key'.")

Connected with API-key authentication.


## Track Created Resources

In [4]:
created_resources = []


def resource_uid(resource):
    if resource is None:
        return None
    if isinstance(resource, str):
        return resource
    if isinstance(resource, dict):
        for key in ("uid", "id", "workspace_id"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("uid", "id", "workspace_id"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_uid(dumped)
    return None


def resource_name(resource):
    if resource is None:
        return None
    if isinstance(resource, dict):
        for key in ("name", "code", "definition", "sampling_feature_code"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("name", "code", "definition", "sampling_feature_code"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_name(dumped)
    return type(resource).__name__


def record_resource(resource_type, resource, station_id=None, source=None):
    created_resources.append({
        "resource_type": resource_type,
        "station_id": station_id,
        "source": source,
        "name_or_code": resource_name(resource),
        "uuid": resource_uid(resource),
        "python_type": type(resource).__name__,
        "resource": resource,
    })
    print(f"{resource_type}: {resource_name(resource)} | uuid={resource_uid(resource)}")
    return resource


def created_resources_dataframe(include_objects=False):
    rows = []
    for row in created_resources:
        rows.append({key: value for key, value in row.items() if include_objects or key != "resource"})
    return pd.DataFrame(rows)

print("Resource registry initialized.")

Resource registry initialized.


## Find or Optionally Create Demo Workspace

In [5]:
workspace = None
workspace_uid = None

if hs_api is None:
    print("Skipping workspace lookup because the HydroServer client is unavailable.")
elif AUTH_METHOD == "anonymous":
    print(f"Anonymous mode: cannot create or manage workspace '{WORKSPACE_NAME}'.")
    print("Switch AUTH_METHOD to 'api_key' for live creation/upload demos.")
else:
    try:
        workspaces = hs_api.workspaces.list(fetch_all=True)
        workspace_items = getattr(workspaces, "items", workspaces)
        workspace = next((item for item in workspace_items if getattr(item, "name", None) == WORKSPACE_NAME), None)
        if workspace is None and CREATE_WORKSPACE_IF_MISSING:
            workspace = record_resource(
                "workspace",
                hs_api.workspaces.create(name=WORKSPACE_NAME, is_private=WORKSPACE_IS_PRIVATE),
            )
        elif workspace is None:
            print(f"Workspace '{WORKSPACE_NAME}' was not found. Ask the facilitator to create it first.")
        else:
            print(f"Using existing workspace: {getattr(workspace, 'name', WORKSPACE_NAME)}")
        workspace_uid = resource_uid(workspace)
        if workspace_uid:
            print(f"Workspace UUID: {workspace_uid}")
    except Exception as exc:
        print(f"Could not find or create workspace '{WORKSPACE_NAME}': {exc}")

Using existing workspace: hydroserver_uganda_demo
Workspace UUID: 019dcc03-9020-718b-9c66-d9da3401eede


## Create Needed Quality-Control Metadata

In [6]:
qc_processing_level_template = {
    "code": f"QC_{demo_run_suffix}",
    "definition": "Quality controlled",
    "explanation": "Workshop quality-control demonstration output.",
    "workspace": workspace_uid or "<workspace-uuid>",
}
qc_result_qualifier_template = {
    "code": f"SUSPECT_{demo_run_suffix}",
    "description": "Flagged by workshop quality-control checks.",
    "workspace": workspace_uid or "<workspace-uuid>",
}
qc_processing_level = None
qc_result_qualifier = None

if hs_api is None or workspace is None or workspace_uid is None:
    print("Skipping QC metadata creation because an authenticated workspace is required.")
else:
    try:
        qc_processing_level = record_resource("processing_level", hs_api.processinglevels.create(**qc_processing_level_template))
        qc_result_qualifier = record_resource("result_qualifier", hs_api.resultqualifiers.create(**qc_result_qualifier_template))
    except Exception as exc:
        print(f"Could not create QC metadata: {exc}")

display(created_resources_dataframe())

processing_level: QC_20260427222118 | uuid=019dd108-9d52-76d8-bfed-ec7c817b3acd
result_qualifier: SUSPECT_20260427222118 | uuid=019dd108-a0b8-7e1c-bb45-acd2eb7a075d


,resource_type,station_id,source,name_or_code,uuid,python_type
0,processing_level,None,None,QC_20260427222118,019dd108-9d52-76d8-bfed-ec7c817b3acd,ProcessingLevel
1,result_qualifier,None,None,SUSPECT_20260427222118,019dd108-a0b8-7e1c-bb45-acd2eb7a075d,ResultQualifier


## Load Observations for Quality Control

In [7]:
observations = pd.read_csv(STREAMFLOW_CSV)
observations["timestamp"] = pd.to_datetime(observations["timestamp"], utc=True)

if hs_api is not None and QUALITY_DATASTREAM_ID:
    try:
        datastream = hs_api.datastreams.get(uid=QUALITY_DATASTREAM_ID)
        hydroserver_observations = datastream.get_observations(include_quality=True, fetch_all=True).dataframe
        print(f"Loaded {len(hydroserver_observations)} observations from HydroServer.")
    except Exception as exc:
        hydroserver_observations = observations.copy()
        print(f"Could not fetch HydroServer observations; using local sample instead: {exc}")
else:
    hydroserver_observations = observations.copy()
    print("Using local sample observations for QC.")

display(hydroserver_observations.head())

Using local sample observations for QC.


,timestamp,value,unit,site,observed_property,quality_note
0,2026-04-01 00:00:00+00:00,42.1,m3/s,Demo River Gauge,streamflow,ok
1,2026-04-01 01:00:00+00:00,43.4,m3/s,Demo River Gauge,streamflow,ok
2,2026-04-01 02:00:00+00:00,44.0,m3/s,Demo River Gauge,streamflow,ok
3,2026-04-01 03:00:00+00:00,NaN,m3/s,Demo River Gauge,streamflow,missing value for QC exercise
4,2026-04-01 04:00:00+00:00,47.2,m3/s,Demo River Gauge,streamflow,ok


## Run QC Checks

In [8]:
qc_frame = hydroserver_observations.copy()
time_column = "phenomenon_time" if "phenomenon_time" in qc_frame.columns else "timestamp"
value_column = "result" if "result" in qc_frame.columns else "value"
qc_frame[time_column] = pd.to_datetime(qc_frame[time_column], utc=True)
qc_frame[value_column] = pd.to_numeric(qc_frame[value_column], errors="coerce")
qc_frame = qc_frame.sort_values(time_column).reset_index(drop=True)
qc_frame["missing_value"] = qc_frame[value_column].isna()
qc_frame["large_gap"] = qc_frame[time_column].diff() > pd.Timedelta(days=2)
qc_frame["suspicious_spike"] = qc_frame[value_column] > qc_frame[value_column].quantile(0.95)
qc_frame["result_qualifier_codes"] = qc_frame.apply(
    lambda row: [qc_result_qualifier_template["code"]] if row["missing_value"] or row["large_gap"] or row["suspicious_spike"] else [],
    axis=1,
)
qc_summary = pd.DataFrame([{
    "rows": len(qc_frame),
    "missing_values": int(qc_frame["missing_value"].sum()),
    "large_gaps": int(qc_frame["large_gap"].sum()),
    "suspicious_spikes": int(qc_frame["suspicious_spike"].sum()),
}])
display(qc_summary)
display(qc_frame[qc_frame["result_qualifier_codes"].map(bool)].head())

,rows,missing_values,large_gaps,suspicious_spikes
0,10,1,0,1


,timestamp,value,unit,site,observed_property,quality_note,missing_value,large_gap,suspicious_spike,result_qualifier_codes
3,2026-04-01 03:00:00+00:00,NaN,m3/s,Demo River Gauge,streamflow,missing value for QC exercise,True,False,False,[SUSPECT_20260427222118]
5,2026-04-01 05:00:00+00:00,138.5,m3/s,Demo River Gauge,streamflow,suspicious spike for QC exercise,False,False,True,[SUSPECT_20260427222118]


## Optional HydroServerQualityControl Pattern

In [9]:
try:
    from hydroserverpy import HydroServerQualityControl
except Exception as exc:
    HydroServerQualityControl = None
    print(f"HydroServerQualityControl is not available in this environment: {exc}")

if HydroServerQualityControl is None or not QUALITY_DATASTREAM_ID:
    print("Skipping HydroServerQualityControl session. Provide QUALITY_DATASTREAM_ID to run it live.")
else:
    hs_quality_control = HydroServerQualityControl(
        datastream_id=QUALITY_DATASTREAM_ID,
        observations=qc_frame,
    )
    hs_quality_control.find_gaps(time_value=1, time_unit="d")
    quality_controlled_observations = hs_quality_control.observations
    display(quality_controlled_observations.head())

Skipping HydroServerQualityControl session. Provide QUALITY_DATASTREAM_ID to run it live.


## Optional Upload of Quality-Controlled Results

In [10]:
quality_controlled_payload = qc_frame[[time_column, value_column, "result_qualifier_codes"]].rename(
    columns={time_column: "phenomenon_time", value_column: "result"}
)

if not UPLOAD_QUALITY_CONTROLLED_RESULTS:
    print("Upload skipped because UPLOAD_QUALITY_CONTROLLED_RESULTS is False.")
elif hs_api is None or not QUALITY_DATASTREAM_ID:
    print("Upload skipped because an authenticated client and target datastream are required.")
else:
    datastream = hs_api.datastreams.get(uid=QUALITY_DATASTREAM_ID)
    datastream.load_observations(quality_controlled_payload)
    print(f"Uploaded {len(quality_controlled_payload)} quality-controlled observations.")

display(quality_controlled_payload.head())

Upload skipped because UPLOAD_QUALITY_CONTROLLED_RESULTS is False.


,phenomenon_time,result,result_qualifier_codes
0,2026-04-01 00:00:00+00:00,42.1,[]
1,2026-04-01 01:00:00+00:00,43.4,[]
2,2026-04-01 02:00:00+00:00,44.0,[]
3,2026-04-01 03:00:00+00:00,NaN,[SUSPECT_20260427222118]
4,2026-04-01 04:00:00+00:00,47.2,[]


## Cleanup: Delete Created Resources

In [ ]:
cleanup_order = [
    "task",
    "data_connection",
    "orchestration_system",
    "datastream",
    "thing",
    "result_qualifier",
    "processing_level",
    "sensor",
    "unit",
    "observed_property",
    "workspace",
]

preview = created_resources_dataframe()
if not preview.empty:
    display(preview)
else:
    print("No created resources are recorded.")

if not DELETE_CREATED_RESOURCES_AT_END:
    print("Cleanup skipped because DELETE_CREATED_RESOURCES_AT_END is False.")
elif hs_api is None:
    print("Cleanup skipped because the HydroServer client is unavailable.")
else:
    deleted_ids = set()
    for resource_type in cleanup_order:
        for row in reversed(created_resources):
            if row["resource_type"] != resource_type:
                continue
            resource = row["resource"]
            uid = resource_uid(resource)
            if resource is None or uid in deleted_ids:
                continue
            try:
                print(f"Deleting {resource_type}: {row['name_or_code']} | uuid={uid}")
                resource.delete()
                deleted_ids.add(uid)
            except Exception as exc:
                print(f"Could not delete {resource_type} {uid}: {exc}")
    print("Cleanup finished.")